# TakeMeter — fine-tuning, baseline, and evaluation

Classifies r/NBA comments into four labels by **what kind of support the take offers**:
`stat_backed`, `consensus_take`, `hot_take`, `reaction`.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**.

**Groq key:** this notebook reads the Colab secret `GROQ_API_KEY` (🔑 in the left sidebar).
Colab secrets are per-account, not per-notebook, so a key you already added for another
notebook is already available here — just make sure *Notebook access* is toggled on.

Run the sections in order. Section 8 writes every file the README needs.

## Section 1 — Setup, label map, upload the dataset

In [ ]:
!pip -q install transformers datasets scikit-learn groq gradio 2>/dev/null

import os, json, time, random, re
import numpy as np, pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABELS = ["stat_backed", "consensus_take", "hot_take", "reaction"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime to T4 GPU")
print("labels:", label2id)

In [ ]:
# Upload data/takemeter_labeled.csv from your repo.
from google.colab import files
up = files.upload()
CSV = list(up.keys())[0]

df = pd.read_csv(CSV, encoding="utf-8-sig")
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].drop_duplicates(subset="text").reset_index(drop=True)

assert set(df["label"]) <= set(LABELS), f"unexpected labels: {set(df['label']) - set(LABELS)}"
print(f"{len(df)} rows")
print(df["label"].value_counts())

### Review gate

`planning.md` §7b says the machine-proposed labels must not be trained on until a human
has reviewed them. This cell enforces that rather than trusting memory. It also reports the
accepted-vs-overridden split, which is the evidence the review actually happened — a very
low override rate is a sign of rubber-stamping, not of good proposals.

In [ ]:
if "pre_labeled" in df.columns:
    status = df["pre_labeled"].value_counts().to_dict()
    print("review status:", status)
    unreviewed = int((df["pre_labeled"] == "proposed").sum())
    if unreviewed:
        raise SystemExit(
            f"STOP: {unreviewed} rows still marked 'proposed'.\n"
            "Run `python3 tools/review_labels.py` locally, finish the review, "
            "re-commit the CSV and re-upload it here."
        )
    reviewed = df[df["pre_labeled"].isin(["accepted", "overridden"])]
    n_over = int((reviewed["pre_labeled"] == "overridden").sum())
    OVERRIDE_RATE = n_over / max(1, len(reviewed))
    print(f"override rate: {n_over}/{len(reviewed)} = {OVERRIDE_RATE:.1%}")
    if "proposed_label" in df.columns:
        flips = (df["label"] != df["proposed_label"])
        print("\nlabel flips during review (proposed -> final):")
        print(pd.crosstab(df.loc[flips, "proposed_label"], df.loc[flips, "label"]))
else:
    OVERRIDE_RATE = None
    print("no pre_labeled column - treating every row as hand-labeled")

## Section 2 — Stratified 70 / 15 / 15 split, then tokenize

Stratified rather than plain random: with ~34 test rows, an unstratified draw can easily
leave a class with two or three test examples, and per-class F1 on three examples is noise.

In [ ]:
from sklearn.model_selection import train_test_split

df["labels"] = df["label"].map(label2id)

train_df, tmp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["labels"])
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, random_state=SEED, stratify=tmp_df["labels"])

for nm, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nm:5s} {len(d):4d}   " +
          "  ".join(f"{l}={int((d['label']==l).sum())}" for l in LABELS))

# Leakage check: identical text must not straddle the split.
overlap = set(train_df["text"]) & set(test_df["text"])
assert not overlap, f"LEAK: {len(overlap)} texts in both train and test"
print("\nno train/test text overlap")

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE = "distilbert-base-uncased"
MAX_LEN = 256   # collection capped comments at 180 words; 256 wordpieces covers that

tokenizer = AutoTokenizer.from_pretrained(BASE)

def to_ds(d):
    ds = Dataset.from_pandas(d[["text", "labels"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True)

ds_train, ds_val, ds_test = to_ds(train_df), to_ds(val_df), to_ds(test_df)

lens = [len(tokenizer(t)["input_ids"]) for t in df["text"]]
print(f"token length: median {int(np.median(lens))}, p95 {int(np.percentile(lens,95))}, "
      f"max {max(lens)} - truncated: {sum(l>MAX_LEN for l in lens)}")

## Section 3 — Fine-tune DistilBERT

**Hyperparameter decision — epochs are chosen by validation macro-F1, not fixed at 3.**
The notebook default of 3 epochs is a guess that ignores the dataset. With ~159 training
examples a single epoch is only 10 optimizer steps, so 3 epochs can leave the model
undertrained, while a fixed 8 will overfit a set this small. Instead this runs up to 8
epochs, evaluates on the validation split after each one, and keeps the checkpoint with the
best macro-F1 (`load_best_model_at_end`), with early stopping after 3 epochs of no gain.
The per-epoch history is printed below and exported, so the README can state which epoch
actually won and what validation loss was doing at that point.

Learning rate stays at 2e-5 and batch size at 16, both standard for BERT-family fine-tuning.
Macro-F1 is the selection metric for the same reason it is the primary metric in
`planning.md` §5: it refuses to let an easy class subsidise a hard one.

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

# transformers renamed evaluation_strategy -> eval_strategy in 4.41; support both.
kw = dict(output_dir="results", learning_rate=2e-5,
          per_device_train_batch_size=16, per_device_eval_batch_size=16,
          num_train_epochs=8, weight_decay=0.01, warmup_ratio=0.1,
          save_total_limit=1, load_best_model_at_end=True,
          metric_for_best_model="macro_f1", greater_is_better=True,
          logging_strategy="epoch", seed=SEED, report_to=[])
try:
    args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **kw)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

trainer.train()

In [ ]:
# Per-epoch history - this is the evidence behind the epochs decision.
hist = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
TRAIN_HISTORY = [{"epoch": round(h["epoch"], 2),
                  "val_loss": round(h["eval_loss"], 4),
                  "val_accuracy": round(h["eval_accuracy"], 4),
                  "val_macro_f1": round(h["eval_macro_f1"], 4)} for h in hist]
tr_loss = {round(h["epoch"],2): round(h["loss"],4)
           for h in trainer.state.log_history if "loss" in h and "eval_loss" not in h}
for h in TRAIN_HISTORY:
    h["train_loss"] = tr_loss.get(h["epoch"])

hist_df = pd.DataFrame(TRAIN_HISTORY)
print(hist_df.to_string(index=False))
BEST_EPOCH = max(TRAIN_HISTORY, key=lambda h: h["val_macro_f1"])["epoch"]
print(f"\nbest epoch by validation macro-F1: {BEST_EPOCH}")

## Section 4 — Evaluate the fine-tuned model on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F

raw = trainer.predict(ds_test)
logits = torch.tensor(raw.predictions)
probs = F.softmax(logits, dim=1).numpy()
ft_pred = probs.argmax(1)
y_true = np.array(test_df["labels"])

FT_ACC = accuracy_score(y_true, ft_pred)
FT_MACRO = f1_score(y_true, ft_pred, average="macro", zero_division=0)
FT_WEIGHTED = f1_score(y_true, ft_pred, average="weighted", zero_division=0)
FT_REPORT = classification_report(y_true, ft_pred, labels=range(len(LABELS)),
                                  target_names=LABELS, output_dict=True, zero_division=0)

print(f"accuracy   {FT_ACC:.3f}")
print(f"macro-F1   {FT_MACRO:.3f}")
print(f"weighted   {FT_WEIGHTED:.3f}\n")
print(classification_report(y_true, ft_pred, labels=range(len(LABELS)),
                            target_names=LABELS, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

FT_CM = confusion_matrix(y_true, ft_pred, labels=range(len(LABELS)))

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(FT_CM, cmap="Blues")
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Fine-tuned DistilBERT - acc {FT_ACC:.2f}, macro-F1 {FT_MACRO:.2f}")
thresh = FT_CM.max() / 2
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, FT_CM[i, j], ha="center", va="center",
                color="white" if FT_CM[i, j] > thresh else "black", fontweight="bold")
fig.colorbar(im, shrink=0.8); fig.tight_layout()
fig.savefig("confusion_matrix.png", dpi=160)
plt.show()

print("\nmarkdown (paste into README):\n")
print("| True \\ Predicted | " + " | ".join(f"`{l}`" for l in LABELS) + " |")
print("|---" * (len(LABELS) + 1) + "|")
for i, l in enumerate(LABELS):
    print(f"| **`{l}`** | " + " | ".join(str(v) for v in FT_CM[i]) + " |")

## Section 5 — Zero-shot baseline: Llama-4-Scout via Groq

Same test set, no task-specific training. The prompt carries the label definitions and the
ordered decision procedure verbatim from `planning.md` §2 — the baseline gets the same rules
the human annotator used, so the comparison is about learning, not about who was told more.

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
GROQ_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

BASELINE_PROMPT = """You are classifying comments from r/NBA by WHAT KIND OF SUPPORT the take offers.

Apply this decision procedure IN ORDER and stop at the first match:

1. Does the comment cite a specific, checkable number (a stat line, percentage, rank, split,
   record, or dated comparison) that does real argumentative work?
   Test: delete the number. If the argument gets weaker, the number was load-bearing.
   If the identical assertion remains with identical force, the number was decorative -
   do NOT stop here, continue to step 2.
   -> stat_backed

2. Otherwise, does it assert an evaluative claim?
   Test: imagine a reply saying "yeah, obviously."
   - Reads as normal, because r/NBA broadly agrees -> consensus_take
   - Reads as sarcastic, because the claim runs against r/NBA consensus -> hot_take
   Consensus means r/NBA consensus, NOT national NBA media consensus. When they disagree,
   r/NBA wins.

3. No evaluative claim survives stripping the emotion (caps, emoji, exclamations, jokes,
   chants like "MVP! MVP!").
   -> reaction

Order matters: a contrarian comment that brings real load-bearing numbers is stat_backed,
not hot_take. Evidence outranks alignment.

Respond with EXACTLY ONE of these four words and nothing else:
stat_backed
consensus_take
hot_take
reaction"""

print(BASELINE_PROMPT)

In [ ]:
def classify_one(text, retries=5):
    for a in range(retries):
        try:
            r = client.chat.completions.create(
                model=GROQ_MODEL, temperature=0, max_tokens=10,
                messages=[{"role": "system", "content": BASELINE_PROMPT},
                          {"role": "user", "content": text}])
            return r.choices[0].message.content.strip()
        except Exception as e:
            if a == retries - 1:
                return f"__ERROR__ {e}"
            time.sleep(2 ** a)

def parse(resp):
    s = (resp or "").strip().lower()
    s = re.sub(r"[^a-z_ ]", " ", s)
    for l in LABELS:                       # exact token first
        if l in s.split():
            return l
    for l in LABELS:                       # then substring
        if l in s.replace(" ", "_"):
            return l
    return None

base_raw, base_parsed = [], []
for i, t in enumerate(test_df["text"].tolist()):
    resp = classify_one(t)
    base_raw.append(resp)
    base_parsed.append(parse(resp))
    print(f"{i+1:3d}/{len(test_df)}  {str(parse(resp)):15s} <- {resp[:40]!r}")
    time.sleep(0.6)   # stay inside the free-tier rate limit

UNPARSEABLE = sum(p is None for p in base_parsed)
print(f"\nunparseable: {UNPARSEABLE}/{len(base_parsed)} ({UNPARSEABLE/len(base_parsed):.1%})")
if UNPARSEABLE / len(base_parsed) > 0.10:
    print("OVER 10% - tighten the prompt's output-format instruction and re-run this cell.")

In [ ]:
# Unparseable responses are scored as wrong rather than dropped: silently dropping them
# would flatter the baseline by removing exactly the cases it handled worst.
MISSING = len(LABELS)   # sentinel class, never a correct answer
base_pred = np.array([label2id[p] if p else MISSING for p in base_parsed])

BASE_ACC = accuracy_score(y_true, base_pred)
BASE_MACRO = f1_score(y_true, base_pred, average="macro", labels=range(len(LABELS)), zero_division=0)
BASE_WEIGHTED = f1_score(y_true, base_pred, average="weighted", labels=range(len(LABELS)), zero_division=0)
BASE_REPORT = classification_report(y_true, base_pred, labels=range(len(LABELS)),
                                    target_names=LABELS, output_dict=True, zero_division=0)
BASE_CM = confusion_matrix(y_true, base_pred, labels=range(len(LABELS)))

print(f"accuracy   {BASE_ACC:.3f}")
print(f"macro-F1   {BASE_MACRO:.3f}")
print(f"weighted   {BASE_WEIGHTED:.3f}\n")
print(classification_report(y_true, base_pred, labels=range(len(LABELS)),
                            target_names=LABELS, zero_division=0))

## Section 6 — Side-by-side comparison, and the success tiers from `planning.md` §6

In [ ]:
maj = test_df["label"].value_counts()
MAJORITY_ACC = maj.iloc[0] / len(test_df)

print(f"{'metric':<14}{'zero-shot':>12}{'fine-tuned':>12}{'delta':>10}")
print("-" * 48)
for nm, b, f in [("accuracy", BASE_ACC, FT_ACC), ("macro-F1", BASE_MACRO, FT_MACRO),
                 ("weighted-F1", BASE_WEIGHTED, FT_WEIGHTED)]:
    print(f"{nm:<14}{b:>12.3f}{f:>12.3f}{f-b:>+10.3f}")
print(f"\nrandom chance {1/len(LABELS):.3f} | always-majority ({maj.index[0]}) {MAJORITY_ACC:.3f}")

print(f"\n{'label':<16}{'base F1':>10}{'ft F1':>10}{'delta':>10}")
print("-" * 46)
for l in LABELS:
    b, f = BASE_REPORT[l]["f1-score"], FT_REPORT[l]["f1-score"]
    print(f"{l:<16}{b:>10.2f}{f:>10.2f}{f-b:>+10.2f}")

In [ ]:
sb = FT_REPORT["stat_backed"]
ch = int(FT_CM[label2id["consensus_take"], label2id["hot_take"]] +
         FT_CM[label2id["hot_take"], label2id["consensus_take"]])
errs = int(FT_CM.sum() - np.trace(FT_CM))
min_f1 = min(FT_REPORT[l]["f1-score"] for l in LABELS)

TIERS = [
  (1, "accuracy >= 0.50", FT_ACC >= 0.50, f"{FT_ACC:.3f}"),
  (1, "macro-F1 >= 0.45", FT_MACRO >= 0.45, f"{FT_MACRO:.3f}"),
  (1, "no class with F1 = 0", min_f1 > 0, f"min per-class F1 {min_f1:.2f}"),
  (1, "beats baseline on macro-F1", FT_MACRO > BASE_MACRO, f"{FT_MACRO:.3f} vs {BASE_MACRO:.3f}"),
  (2, "accuracy >= 0.70", FT_ACC >= 0.70, f"{FT_ACC:.3f}"),
  (2, "macro-F1 >= 0.65", FT_MACRO >= 0.65, f"{FT_MACRO:.3f}"),
  (2, "every per-class F1 >= 0.55", min_f1 >= 0.55, f"min {min_f1:.2f}"),
  (2, "consensus<->hot under half of errors", (ch < errs/2) if errs else True,
      f"{ch} of {errs} errors"),
  (3, "stat_backed precision >= 0.80", sb["precision"] >= 0.80, f"{sb['precision']:.2f}"),
  (3, "stat_backed recall >= 0.50", sb["recall"] >= 0.50, f"{sb['recall']:.2f}"),
]
for t, crit, ok, actual in TIERS:
    print(f"  T{t}  {'PASS' if ok else 'MISS'}  {crit:<38} {actual}")

if FT_ACC > 0.95:
    print("\n>0.95 - planning.md §6 says treat this as a red flag: check for leakage, "
          "near-duplicates across the split, and trivially separable labels.")

## Section 7 — Confidence calibration *(stretch)*

Does a 90%-confident prediction actually get it right more often than a 60%-confident one?
Reported two ways: accuracy within confidence buckets, and **expected calibration error**
(the average gap between confidence and accuracy, weighted by bucket size).

In [ ]:
conf = probs.max(1)
correct = (ft_pred == y_true)

edges = [0.0, 0.5, 0.7, 0.85, 0.95, 1.001]
names = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
CALIB = []
for lo, hi, nm in zip(edges[:-1], edges[1:], names):
    m = (conf >= lo) & (conf < hi)
    if m.sum():
        CALIB.append({"bucket": nm, "n": int(m.sum()),
                      "mean_confidence": round(float(conf[m].mean()), 3),
                      "accuracy": round(float(correct[m].mean()), 3)})

ECE = float(sum(b["n"] * abs(b["mean_confidence"] - b["accuracy"]) for b in CALIB) / len(conf))
print(pd.DataFrame(CALIB).to_string(index=False))
print(f"\nexpected calibration error: {ECE:.3f}")
print(f"mean confidence {conf.mean():.3f} vs accuracy {FT_ACC:.3f} "
      f"-> {'OVERconfident' if conf.mean() > FT_ACC else 'UNDERconfident'}")
print(f"\ncorrect  mean conf {conf[correct].mean():.3f}")
print(f"wrong    mean conf {conf[~correct].mean():.3f}")

print("\nmarkdown:\n")
print("| Confidence bucket | n | Mean confidence | Accuracy |")
print("|---|---|---|---|")
for b in CALIB:
    print(f"| {b['bucket']} | {b['n']} | {b['mean_confidence']:.2f} | {b['accuracy']:.2f} |")

## Section 8 — Export everything the README needs

Writes `evaluation_results.json` (headline metrics, per-class, both confusion matrices,
training history, calibration) and `test_predictions.csv` (every test comment with its true
label, both models' predictions, and the fine-tuned confidence). The second file is what the
error-pattern analysis and the sample-classifications table are built from.

In [ ]:
pred_rows = []
for k, (_, r) in enumerate(test_df.reset_index(drop=True).iterrows()):
    pred_rows.append({
        "text": r["text"],
        "true_label": r["label"],
        "ft_pred": LABELS[ft_pred[k]],
        "ft_confidence": round(float(conf[k]), 4),
        "ft_correct": bool(correct[k]),
        "base_pred": base_parsed[k] or "UNPARSEABLE",
        "base_raw": base_raw[k],
        "base_correct": bool(base_parsed[k] == r["label"]),
        "word_count": len(str(r["text"]).split()),
        "source_thread_type": r.get("source_thread_type", ""),
        "review_flag": r.get("review_flag", ""),
        "notes": r.get("notes", ""),
        **{f"p_{l}": round(float(probs[k][i]), 4) for i, l in enumerate(LABELS)},
    })
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv("test_predictions.csv", index=False)

results = {
    "labels": LABELS,
    "dataset": {
        "total": int(len(df)),
        "distribution": df["label"].value_counts().to_dict(),
        "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
        "override_rate": OVERRIDE_RATE,
    },
    "training": {
        "base_model": BASE, "epochs_max": 8, "best_epoch": BEST_EPOCH,
        "learning_rate": 2e-5, "batch_size": 16, "max_length": MAX_LEN,
        "weight_decay": 0.01, "warmup_ratio": 0.1, "seed": SEED,
        "selection_metric": "validation macro-F1",
        "history": TRAIN_HISTORY,
    },
    "baseline": {
        "model": GROQ_MODEL, "temperature": 0,
        "unparseable": int(UNPARSEABLE), "n_test": int(len(test_df)),
        "accuracy": round(BASE_ACC, 4), "macro_f1": round(BASE_MACRO, 4),
        "weighted_f1": round(BASE_WEIGHTED, 4),
        "per_class": {l: {k: round(v, 4) for k, v in BASE_REPORT[l].items()} for l in LABELS},
        "confusion_matrix": BASE_CM.tolist(),
        "prompt": BASELINE_PROMPT,
    },
    "fine_tuned": {
        "accuracy": round(FT_ACC, 4), "macro_f1": round(FT_MACRO, 4),
        "weighted_f1": round(FT_WEIGHTED, 4),
        "per_class": {l: {k: round(v, 4) for k, v in FT_REPORT[l].items()} for l in LABELS},
        "confusion_matrix": FT_CM.tolist(),
    },
    "reference_points": {
        "random_chance": round(1 / len(LABELS), 4),
        "always_majority": round(float(MAJORITY_ACC), 4),
        "majority_label": maj.index[0],
    },
    "calibration": {"buckets": CALIB, "ece": round(ECE, 4),
                    "mean_confidence": round(float(conf.mean()), 4),
                    "mean_conf_correct": round(float(conf[correct].mean()), 4),
                    "mean_conf_wrong": round(float(conf[~correct].mean()), 4)},
    "success_tiers": [{"tier": t, "criterion": c, "met": bool(ok), "actual": a}
                      for t, c, ok, a in TIERS],
}
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"wrote evaluation_results.json and test_predictions.csv "
      f"({len(pred_df)} rows, {int((~pred_df.ft_correct).sum())} fine-tuned errors)")

In [ ]:
# Save the model so the Gradio app in Section 9 can load it, then download everything.
trainer.save_model("takemeter_model"); tokenizer.save_pretrained("takemeter_model")
!zip -qr takemeter_model.zip takemeter_model

from google.colab import files
for fn in ["evaluation_results.json", "test_predictions.csv", "confusion_matrix.png"]:
    files.download(fn)
print("also available to download manually: takemeter_model.zip (~250MB)")

## Section 9 — Live interface *(stretch)*

Paste a comment, get the predicted label and confidence. Running this cell prints a public
`gradio.live` URL — that is the interface to film for the demo video. The same code is
committed to the repo as `tools/takemeter_app.py` and runs locally against an unzipped
`takemeter_model/`.

In [ ]:
import gradio as gr
from transformers import pipeline

clf = pipeline("text-classification", model="takemeter_model",
               tokenizer="takemeter_model", top_k=None,
               device=0 if torch.cuda.is_available() else -1)

BLURB = {
    "stat_backed":    "cites a checkable number that does argumentative work",
    "consensus_take": "unsupported claim r/NBA broadly agrees with",
    "hot_take":       "unsupported claim that runs against r/NBA consensus",
    "reaction":       "in-the-moment emotion, joke or chant - no claim survives",
}

def classify(text):
    if not text.strip():
        return {}, ""
    scores = {d["label"]: float(d["score"]) for d in clf(text[:2000])[0]}
    top = max(scores, key=scores.get)
    return scores, f"**{top}** at {scores[top]:.1%} confidence - {BLURB[top]}"

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=5, label="r/NBA comment",
                      placeholder="Paste a comment..."),
    outputs=[gr.Label(num_top_classes=4, label="Confidence across all four labels"),
             gr.Markdown()],
    title="TakeMeter",
    description="Fine-tuned DistilBERT sorting r/NBA comments by what kind of support the take offers.",
    examples=[
        ["He averaged 27.4 on 61% TS after the All-Star break in 28 games, that's a better stretch than his MVP year."],
        ["Jokic is the best passing big man of all time and it isn't particularly close."],
        ["Curry is the most overrated player in league history, it's not even debatable."],
        ["BRO WHAT WAS THAT 😭😭 no shot he actually hit that"],
        ["LeBron is overrated, his playoff record against 1-seeds is under .500."],
    ],
    flagging_mode="never" if hasattr(gr, "__version__") and gr.__version__ >= "5" else None,
)
demo.launch(share=True, debug=False)